<h2>text processing — tokenization, stemming, lemmatization</h2>

<h3>building a regex word tokenizer</h3>

In [1]:
import re 

def tokenize(text): 
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\sA-Za-z0-9]", text)

In [3]:
tokenize("We are all stardust and stories.")

['We', 'are', 'all', 'stardust', 'and', 'stories', '.']

<h3>building a porter stemmer</h3>

In [7]:
def stem(word):
    if word.endswith("sses"):
        return word[:-2]
    if word.endswith("ies"):
        return word[:-2]
    if word.endswith("ss"):
        return word
    if word.endswith("s") and len(word) > 1:
        return word[:-1]
    return word

In [13]:
[stem(w) for w in ["caresses", "ponies", "caress", "cats", "bus"]]

['caress', 'poni', 'caress', 'cat', 'bu']

<h3>building a lookup-based lemmatizer</h3>

In [17]:
#lemmatization reduces a word to its dictionary form using grammar knowledge 
LEMMA_TABLE = {
    ("running", "VERB"): "run",
    ("ran", "VERB"): "run",
    ("runs", "VERB"): "run",
    ("better", "ADJ"): "good",
    ("best", "ADJ"): "good",
    ("cats", "NOUN"): "cat",
    ("cat", "NOUN"): "cat",
    ("were", "VERB"): "be",
    ("was", "VERB"): "be",
    ("is", "VERB"): "be",
}

def lemmatize(word, pos):
    key = (word.lower(), pos)
    if key in LEMMA_TABLE:
        return LEMMA_TABLE[key]
    if pos == "VERB" and word.endswith("ing"):
        return word[:-3]
    if pos == "NOUN" and word.endswith("s"):
        return word[:-1]
    return word.lower()

In [21]:
lemmatize("running", "VERB") 

'run'

In [23]:
lemmatize("cats", "NOUN")   

'cat'

In [27]:
lemmatize("better", "ADJ")  

'good'

In [29]:
lemmatize("watched", "VERB")

'watched'

In [31]:
def preprocess(text, pos_tagger=None):
    tokens = tokenize(text)
    stems = [stem(t.lower()) for t in tokens]
    tags = pos_tagger(tokens) if pos_tagger else [(t, "NOUN") for t in tokens]
    lemmas = [lemmatize(word, pos) for word, pos in tags]
    return {"tokens": tokens, "stems": stems, "lemmas": lemmas}

<h3>using NLTK</h3>

In [40]:
import nltk
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")

from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

text = "Occasionally, Fate pulls itself together again and Time is always waiting."
tokens = word_tokenize(text)
stems = [PorterStemmer().stem(t) for t in tokens]
lemmatizer = WordNetLemmatizer()
tagged = pos_tag(tokens)


def nltk_pos_to_wordnet(tag):
    if tag.startswith("V"):
        return "v"
    if tag.startswith("J"):
        return "a"
    if tag.startswith("R"):
        return "r"
    return "n"


lemmas = [lemmatizer.lemmatize(t, nltk_pos_to_wordnet(tag)) for t, tag in tagged]

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LALITHA\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LALITHA\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\LALITHA\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


<h3>using spaCy</h3>

In [52]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Occasionally, Fate pulls itself together again and Time is always waiting.")

for token in doc:
    print(token.text, token.lemma_, token.pos_)

Occasionally occasionally ADV
, , PUNCT
Fate Fate PROPN
pulls pull VERB
itself itself PRON
together together ADV
again again ADV
and and CCONJ
Time Time PROPN
is be AUX
always always ADV
waiting wait VERB
. . PUNCT


<h2>text representation</h2>

<h3>building a bag of words & tf-idf</h3>

In [58]:
def build_vocab(docs):
    vocab = {}
    for doc in docs:
        for token in doc:
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab 

def bag_of_words(docs, vocab):
    matrix = [[0] * len(vocab) for _ in docs]
    for i, doc in enumerate(docs):
        for token in doc:
            if token in vocab:
                matrix[i][vocab[token]] += 1
    return matrix

In [66]:
docs = [["dog", "hell", "oh", "other"], ["dog", "dog", "dear"]] 
vocab = build_vocab(docs) 

#[i][j] - how many times word j appears in doc i 
bag_of_words(docs, vocab)

[[1, 1, 1, 1, 0], [2, 0, 0, 0, 1]]

In [68]:
import math

#term frequency = word count/total no. of words 
def term_frequency(doc_bow, doc_length):
    return [c / doc_length if doc_length else 0 for c in doc_bow]

#document frequency = in how many docs this word appears 
def document_frequency(bow_matrix):
    df = [0] * len(bow_matrix[0])
    for row in bow_matrix:
        for j, count in enumerate(row):
            if count > 0:
                df[j] += 1
    return df

#inverse document frequency = log(total docs/docs which have the word) 
#common words - low importance, rare words - high importance 
def inverse_document_frequency(df, n_docs):
    return [math.log((n_docs + 1) / (d + 1)) + 1 for d in df] 

#freq in this doc & rare in other docs = imp word 
def tfidf(bow_matrix):
    n_docs = len(bow_matrix)
    df = document_frequency(bow_matrix)
    idf = inverse_document_frequency(df, n_docs)
    out = []
    for row in bow_matrix:
        length = sum(row)
        tf = term_frequency(row, length)
        out.append([tf_j * idf_j for tf_j, idf_j in zip(tf, idf)])
    return out

In [71]:
docs = [["the", "dog", "blue"],
        ["the", "eyes", "blue"],
        ["the", "dog", "horse"],] 
vocab = build_vocab(docs) 
bow = bag_of_words(docs, vocab) 
tfidf(bow)

[[0.3333333333333333, 0.42922735748392693, 0.42922735748392693, 0.0, 0.0],
 [0.3333333333333333, 0.0, 0.42922735748392693, 0.5643823935199818, 0.0],
 [0.3333333333333333, 0.42922735748392693, 0.0, 0.0, 0.5643823935199818]]

In [73]:
def l2_normalize(matrix):
    out = []
    for row in matrix:
        norm = math.sqrt(sum(x * x for x in row))
        out.append([x / norm if norm else 0 for x in row])
    return out

<h3>hybrid - TF-IDF weighted embeddings</h3>

In [79]:
def tfidf_weighted_embedding(doc, tfidf_scores, embedding_table, dim):
    vec = [0.0] * dim
    total_weight = 0.0
    for token in doc:
        if token not in embedding_table or token not in tfidf_scores:
            continue
        weight = tfidf_scores[token]
        emb = embedding_table[token]
        for i in range(dim):
            vec[i] += weight * emb[i]
        total_weight += weight
    if total_weight == 0:
        return vec
    return [v / total_weight for v in vec]

<h3>using scikit-learn's built-ins</h3>  

In [76]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = ["the dog sang on the stage", "the horse sang on the stage", "the dog danced"]

bow_vectorizer = CountVectorizer()
bow = bow_vectorizer.fit_transform(docs)
print(bow_vectorizer.get_feature_names_out())
print(bow.toarray())

tfidf_vectorizer = TfidfVectorizer()
tfidf = tfidf_vectorizer.fit_transform(docs)
print(tfidf.toarray().round(3))

['danced' 'dog' 'horse' 'on' 'sang' 'stage' 'the']
[[0 1 0 1 1 1 2]
 [0 0 1 1 1 1 2]
 [1 1 0 0 0 0 1]]
[[0.    0.395 0.    0.395 0.395 0.395 0.613]
 [0.    0.    0.492 0.374 0.374 0.374 0.581]
 [0.72  0.548 0.    0.    0.    0.    0.425]]


<h2>word embeddings</h2>

<h3>training pairs from corpus</h3>

In [3]:
def skipgram_pairs(docs, window=2):
    pairs = []
    for doc in docs:
        for i, center in enumerate(doc):
            for j in range(max(0, i - window), min(len(doc), i + window + 1)):
                if i == j:
                    continue
                pairs.append((center, doc[j]))
    return pairs

In [5]:
skipgram_pairs([["the", "cat", "sat", "on", "mat"]], window=2)

[('the', 'cat'),
 ('the', 'sat'),
 ('cat', 'the'),
 ('cat', 'sat'),
 ('cat', 'on'),
 ('sat', 'the'),
 ('sat', 'cat'),
 ('sat', 'on'),
 ('sat', 'mat'),
 ('on', 'cat'),
 ('on', 'sat'),
 ('on', 'mat'),
 ('mat', 'sat'),
 ('mat', 'on')]

<h3>embedding tables </h3>

In [9]:
#vocab_size - how many words in dictionary 
#dim - how many random numbers for each word 
import numpy as np

def init_embeddings(vocab_size, dim, seed=0):
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.1, size=(vocab_size, dim))
    W_prime = rng.normal(0, 0.1, size=(vocab_size, dim))
    return W, W_prime

<h3>negative sampling objective</h3>

In [29]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))


def train_pair(W, W_prime, center_idx, context_idx, negative_indices, lr):
    #secret numbers for center word 
    v_c = W[center_idx]
    #secret numbers for correct partner word 
    u_pos = W_prime[context_idx]
    #secret numbers for wrong partner word 
    u_negs = W_prime[negative_indices]

    #how close correct partner words match 
    pos_score = sigmoid(v_c @ u_pos) 
    #how far wrong partner words are
    neg_scores = sigmoid(u_negs @ v_c)

    #how much center word needs to change to get closer to corrcet word and farther from wrong word 
    grad_center = (pos_score - 1) * u_pos
    for i, u in enumerate(u_negs):
        grad_center += neg_scores[i] * u

    W[context_idx] = W[context_idx]
    W_prime[context_idx] -= lr * (pos_score - 1) * v_c
    for i, neg_idx in enumerate(negative_indices):
        W_prime[neg_idx] -= lr * neg_scores[i] * v_c
    W[center_idx] -= lr * grad_center

<h3>train on a toy corpus</h3>

In [31]:
def train(docs, dim=16, window=2, k_neg=5, epochs=100, lr=0.05, seed=0):
    vocab = build_vocab(docs)
    vocab_size = len(vocab)
    rng = np.random.default_rng(seed)
    W, W_prime = init_embeddings(vocab_size, dim, seed=seed)
    pairs = skipgram_pairs(docs, window=window)

    for epoch in range(epochs):
        rng.shuffle(pairs)
        for center, context in pairs:
            c_idx = vocab[center]
            ctx_idx = vocab[context]
            negs = rng.integers(0, vocab_size, size=k_neg)
            negs = [n for n in negs if n != ctx_idx and n != c_idx]
            train_pair(W, W_prime, c_idx, ctx_idx, negs, lr)
    return vocab, W

<h3>similarity & analogy</h3>

In [33]:
#which words in the dictionary have secret numbers most similar to this target 
def nearest(vocab, W, target_vec, topk=5, exclude=None):
    exclude = exclude or set()
    inv_vocab = {i: w for w, i in vocab.items()}
    norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-9
    W_norm = W / norms
    target = target_vec / (np.linalg.norm(target_vec) + 1e-9)
    sims = W_norm @ target
    order = np.argsort(-sims)
    out = []
    for i in order:
        if i in exclude:
            continue
        out.append((inv_vocab[i], float(sims[i])))
        if len(out) == topk:
            break
    return out

#classic word relationship puzzle 
def analogy(vocab, W, a, b, c, topk=5):
    v = W[vocab[b]] - W[vocab[a]] + W[vocab[c]]
    return nearest(vocab, W, v, topk=topk, exclude={vocab[a], vocab[b], vocab[c]})

In [37]:
analogy(vocab, W, "man", "king", "woman") 

NameError: name 'vocab' is not defined

<h3>using gensim</h3>

In [40]:
from gensim.models import Word2Vec

sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "ran", "across", "the", "room"],
]

model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    negative=5,
    workers=4,
    epochs=30,
)

print(model.wv["cat"])
print(model.wv.most_similar("cat", topn=3))

[-9.5785465e-03  8.9431154e-03  4.1650687e-03  9.2347348e-03
  6.6435025e-03  2.9247368e-03  9.8040197e-03 -4.4246409e-03
 -6.8033109e-03  4.2273807e-03  3.7290000e-03 -5.6646108e-03
  9.7047603e-03 -3.5583067e-03  9.5494064e-03  8.3472609e-04
 -6.3384566e-03 -1.9771170e-03 -7.3770545e-03 -2.9795230e-03
  1.0416972e-03  9.4826873e-03  9.3558477e-03 -6.5958775e-03
  3.4751510e-03  2.2755705e-03 -2.4893521e-03 -9.2291720e-03
  1.0271263e-03 -8.1657059e-03  6.3201892e-03 -5.8000805e-03
  5.5354391e-03  9.8337233e-03 -1.6000033e-04  4.5284927e-03
 -1.8094003e-03  7.3607611e-03  3.9400971e-03 -9.0103243e-03
 -2.3985039e-03  3.6287690e-03 -9.9568366e-05 -1.2012708e-03
 -1.0554385e-03 -1.6716016e-03  6.0495257e-04  4.1650953e-03
 -4.2527914e-03 -3.8336217e-03 -5.2816868e-05  2.6935578e-04
 -1.6880632e-04 -4.7855065e-03  4.3134023e-03 -2.1719194e-03
  2.1035396e-03  6.6652300e-04  5.9696771e-03 -6.8423809e-03
 -6.8157101e-03 -4.4762576e-03  9.4358288e-03 -1.5918827e-03
 -9.4292425e-03 -5.45041

<h2>GloVe</h2>

In [54]:
#similar to word2vec 

In [43]:
import numpy as np
from collections import Counter

#every word gets its own id, neighbours get points 
def build_cooccurrence(docs, window=5):
    pair_counts = Counter()
    vocab = {}
    for doc in docs:
        for token in doc:
            if token not in vocab:
                vocab[token] = len(vocab)
    for doc in docs:
        indexed = [vocab[t] for t in doc]
        for i, center in enumerate(indexed):
            for j in range(max(0, i - window), min(len(indexed), i + window + 1)):
                if i != j:
                    distance = abs(i - j)
                    pair_counts[(center, indexed[j])] += 1.0 / distance
    return vocab, pair_counts

#using the tally to assign words secret numbers 
def glove_train(vocab, pair_counts, dim=16, epochs=100, lr=0.05, x_max=100, alpha=0.75, seed=0):
    n = len(vocab)
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.1, size=(n, dim))
    W_tilde = rng.normal(0, 0.1, size=(n, dim))
    b = np.zeros(n)
    b_tilde = np.zeros(n)

    for epoch in range(epochs):
        for (i, j), x_ij in pair_counts.items():
            weight = (x_ij / x_max) ** alpha if x_ij < x_max else 1.0
            diff = W[i] @ W_tilde[j] + b[i] + b_tilde[j] - np.log(x_ij)
            coef = weight * diff

            grad_W_i = coef * W_tilde[j]
            grad_W_tilde_j = coef * W[i]
            W[i] -= lr * grad_W_i
            W_tilde[j] -= lr * grad_W_tilde_j
            b[i] -= lr * coef
            b_tilde[j] -= lr * coef

    return W + W_tilde

<h2>FastText</h2>

In [52]:
#breaks down words into little little pieces, if there is a word that has never been 
# encountered before then that word is broken down into pieces and vector sum calculated 
# that sum lands somewhere near known words 
# e.g. catlike lands somewhere near cat and feline 

In [46]:
def char_ngrams(word, n_min=3, n_max=6):
    wrapped = f"<{word}>"
    grams = {wrapped}
    for n in range(n_min, n_max + 1):
        for i in range(len(wrapped) - n + 1):
            grams.add(wrapped[i:i + n])
    return grams

In [48]:
char_ngrams("where")

{'<wh',
 '<whe',
 '<wher',
 '<where',
 '<where>',
 'ere',
 'ere>',
 'her',
 'here',
 'here>',
 're>',
 'whe',
 'wher',
 'where',
 'where>'}

In [50]:
def fasttext_vector(word, ngram_table):
    grams = char_ngrams(word)
    vecs = [ngram_table[g] for g in grams if g in ngram_table]
    if not vecs:
        return None
    return np.sum(vecs, axis=0)

<h2>BPE - learned subword vocab </h2>

In [57]:
#rare or long words - broken down into clearr chunks - un for get table 
# common words are chunks - the 
# new words are read letter by letter 

In [59]:
#all words are broken letter by letter, end is marked by </w> 
def learn_bpe(corpus, k_merges):
    vocab = Counter()
    for word, freq in corpus.items():
        tokens = tuple(word) + ("</w>",)
        vocab[tokens] = freq

    merges = []
    for _ in range(k_merges): 
        #most popular letter pairs are grouped together like th 
        pair_freq = Counter()
        for tokens, freq in vocab.items():
            for a, b in zip(tokens, tokens[1:]):
                pair_freq[(a, b)] += freq
        if not pair_freq:
            break
        best = pair_freq.most_common(1)[0][0]
        merges.append(best)

        new_vocab = Counter()
        for tokens, freq in vocab.items():
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == best:
                    new_tokens.append(tokens[i] + tokens[i + 1])
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            new_vocab[tuple(new_tokens)] = freq
        vocab = new_vocab
    return merges

#when new word is given, break into individual letters and 
# then check from learned rules if there is anything that can be grouped together 
# return final chunks 
def apply_bpe(word, merges):
    tokens = list(word) + ["</w>"]
    for a, b in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

In [61]:
import fasttext.util
fasttext.util.download_model("en", if_exists="ignore")
ft = fasttext.load_model("cc.en.300.bin")
print(ft.get_word_vector("whereupon").shape)
print(ft.get_word_vector("zoomerapproved").shape)

ModuleNotFoundError: No module named 'fasttext'

In [63]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
print(tok.tokenize("unbelievably tokenized"))

ModuleNotFoundError: No module named 'transformers'